# Haar and Walsh Transforms

## Haar wavelet transform

The **Haar wavelet** is the simplest orthonormal wavelet. The 1D Haar system on $[0,1]$ consists of:
- **Scaling function**: $\phi(x) = \mathbf{1}_{[0,1]}$.
- **Mother wavelet**: $\psi(x) = \mathbf{1}_{[0,1/2)} - \mathbf{1}_{[1/2,1)}$.
- **Dilated/translated wavelets**: $\psi_{j,k}(x) = 2^{j/2}\psi(2^j x - k)$, $j \ge 0$, $0 \le k < 2^j$.

For a signal of length $N = 2^J$, the fast Haar transform proceeds level by level:
$$
a_{j-1}[k] = \frac{a_j[2k] + a_j[2k+1]}{\sqrt{2}}, \qquad d_{j-1}[k] = \frac{a_j[2k] - a_j[2k+1]}{\sqrt{2}}.
$$

## Walsh–Hadamard transform

The **Walsh functions** are $\{\pm 1\}$-valued analogues of sinusoids, indexed by **sequency** (number of sign changes). The **Hadamard matrix** $H_n$ (size $2^n \times 2^n$) is defined recursively:
$$
H_0 = [1], \qquad H_n = \frac{1}{\sqrt{2}} \begin{pmatrix} H_{n-1} & H_{n-1} \\ H_{n-1} & -H_{n-1} \end{pmatrix}.
$$
The WHT is $\hat{x} = H_n x$. It is its own inverse: $H_n^2 = I$.

## Applications

- Haar: multiresolution image compression (JPEG 2000 uses more sophisticated wavelets, but Haar is the prototype).
- Walsh: fast Boolean function analysis, CDMA spreading codes, quantum computing (Hadamard gate).

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

plt.rcParams['figure.dpi'] = 120

## Haar transform implementation

We implement the fast Haar transform and its inverse.

In [ ]:
def haar_fwd(x):
    """Fast Haar transform of signal x (length must be power of 2)."""
    x = x.copy().astype(float)
    n = len(x)
    result = np.zeros(n)
    while n > 1:
        half = n // 2
        a = (x[:n:2] + x[1:n:2]) / np.sqrt(2)
        d = (x[:n:2] - x[1:n:2]) / np.sqrt(2)
        result[half:n] = d
        x[:half] = a
        n = half
    result[0] = x[0]
    return result

def haar_inv(c):
    """Inverse Haar transform."""
    c = c.copy().astype(float)
    n = 1
    N = len(c)
    x = np.array([c[0]])
    while n < N:
        half = n
        n = 2 * half
        a = x[:half]
        d = c[half:n]
        x = np.empty(n)
        x[0::2] = (a + d) / np.sqrt(2)
        x[1::2] = (a - d) / np.sqrt(2)
    return x

def hadamard_matrix(n):
    """Hadamard matrix of size 2^n x 2^n (unnormalized)."""
    H = np.array([[1.]])
    for _ in range(n):
        H = np.block([[H, H], [H, -H]])
    return H / np.sqrt(2**n)

N = 64
J = int(np.log2(N))

# Test orthogonality
H = hadamard_matrix(J)
print(f'Hadamard orthogonal: {np.allclose(H @ H.T, np.eye(N))}')

c_test = haar_fwd(np.random.default_rng(0).standard_normal(N))
print(f'Haar roundtrip: {np.allclose(haar_inv(c_test), np.random.default_rng(0).standard_normal(N))}')

## Haar basis functions and Walsh functions

We visualise the first 16 Haar basis vectors and the first 16 Walsh (Hadamard) basis vectors.

In [ ]:
N_show = 16
fig, axes = plt.subplots(2, N_show//4, figsize=(14, 5))

for i, ax in enumerate(axes[0]):
    e = np.zeros(N_show); e[i] = 1.0
    h = haar_inv(e)
    ax.step(np.linspace(0, 1, N_show+1)[:-1], h, where='post', color='royalblue', lw=1.5)
    ax.axhline(0, color='k', lw=0.5); ax.axis('off')
    ax.set_title(f'$\\psi_{{{i}}}$', fontsize=8)

H16 = hadamard_matrix(int(np.log2(N_show)))
for i, ax in enumerate(axes[1]):
    ax.step(np.linspace(0, 1, N_show+1)[:-1], H16[i], where='post', color='tomato', lw=1.5)
    ax.axhline(0, color='k', lw=0.5); ax.axis('off')
    ax.set_title(f'$w_{{{i}}}$', fontsize=8)

axes[0][0].set_ylabel('Haar', fontsize=9)
axes[1][0].set_ylabel('Walsh', fontsize=9)
fig.suptitle('Haar basis (top) and Walsh–Hadamard basis (bottom)', y=1.02)
plt.tight_layout(); plt.show()

## Multiresolution analysis: signal decomposition

We decompose a piecewise-smooth signal and visualise the approximation and detail coefficients at each level.

In [ ]:
N_sig = 128
J_sig = int(np.log2(N_sig))
x_sig = np.zeros(N_sig)
t = np.arange(N_sig) / N_sig
x_sig = np.sin(4*np.pi*t) + 0.5*(t > 0.4)*(t < 0.7) + 0.2*np.random.default_rng(1).standard_normal(N_sig)

c_haar = haar_fwd(x_sig)

fig, axes = plt.subplots(J_sig//2, 2, figsize=(12, 8))
for level in range(J_sig//2):
    start = N_sig // (2**(level+1))
    end   = N_sig // (2**level)
    coeffs = c_haar[start:end]
    axes[level, 0].bar(np.arange(len(coeffs)) + start, np.abs(coeffs), color='seagreen', alpha=0.8)
    axes[level, 0].set_ylabel(f'Level {J_sig-level-1}', fontsize=8)
    axes[level, 0].set_xlim(0, N_sig); axes[level, 0].grid(alpha=0.2)
    # Reconstruction from this level only
    c_level = np.zeros(N_sig)
    c_level[0] = c_haar[0]
    c_level[start:end] = c_haar[start:end]
    x_rec = haar_inv(c_level)
    axes[level, 1].plot(x_sig, 'k-', lw=0.8, alpha=0.4)
    axes[level, 1].plot(x_rec, color='royalblue', lw=1.5)
    axes[level, 1].set_xlim(0, N_sig); axes[level, 1].grid(alpha=0.2)

axes[0][0].set_title('Detail coefficients'); axes[0][1].set_title('Approx + one level')
plt.tight_layout(); plt.show()

## Interactive: hard thresholding for denoising

In [ ]:
rng = np.random.default_rng(7)
x_clean = np.sin(4*np.pi*t) + 0.5*(t>0.4)*(t<0.7)
x_noisy = x_clean + 0.3 * rng.standard_normal(N_sig)

def show_haar_denoise(threshold=0.3, transform='Haar'):
    if transform == 'Haar':
        c = haar_fwd(x_noisy)
        c_thr = c * (np.abs(c) > threshold)
        x_rec = haar_inv(c_thr)
    else:
        H_full = hadamard_matrix(J_sig)
        c = H_full @ x_noisy
        c_thr = c * (np.abs(c) > threshold)
        x_rec = H_full @ c_thr
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(x_noisy, 'gray', lw=0.8, alpha=0.5, label='noisy')
    axes[0].plot(x_clean, 'k--', lw=1.5, label='clean')
    axes[0].plot(x_rec, 'royalblue', lw=2, label=f'{transform} denoised')
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.2)
    axes[0].set_title(f'{transform} denoising (thr={threshold:.2f})')
    if transform == 'Haar': c_full = haar_fwd(x_noisy)
    else: c_full = H_full @ x_noisy
    axes[1].bar(np.arange(N_sig), np.abs(c_full), color='steelblue', alpha=0.6)
    axes[1].axhline(threshold, color='tomato', lw=2, ls='--', label='threshold')
    axes[1].legend(); axes[1].set_xlabel('index'); axes[1].set_ylabel('|coefficient|')
    plt.tight_layout(); plt.show()

from ipywidgets import Dropdown
interact(show_haar_denoise,
         threshold=FloatSlider(value=0.3, min=0.0, max=1.5, step=0.05, description='threshold'),
         transform=Dropdown(options=['Haar', 'Walsh'], description='transform'));

## Bibliographical resources

- Haar, A. (1910). Zur Theorie der orthogonalen Funktionensysteme. *Mathematische Annalen*, 69(3), 331–371.
- Walsh, J. L. (1923). A closed set of normal orthogonal functions. *American Journal of Mathematics*, 45(1), 5–24.
- Mallat, S. (2009). *A Wavelet Tour of Signal Processing: The Sparse Way* (3rd ed.). Academic Press.
- Daubechies, I. (1992). *Ten Lectures on Wavelets*. SIAM.
- Ahmed, N. and Rao, K. R. (1975). *Orthogonal Transforms for Digital Signal Processing*. Springer.